# Inspect resulting DB, including duplicates

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
        if table == "fame_derived":
            # Fetch all the unique values in the columns industry_codes
            industry_codes = con.table(table).select("industry_codes").distinct().execute()
            industry_codes_sorted = sorted(industry_codes["industry_codes"].dropna().unique())
            f.write(f"### Unique industry_codes in {table}: {", ".join(industry_codes_sorted)}")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duckdb_tables.md


## [read] Check for conflicts in fame_yearly

In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

sample_size = 2000
table_name = "fame_yearly"

db_path = dirs.output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(str(db_path))
t = con.table(table_name)
keys = ["registered_number", "year"]
print(f"🔍 Sampling {sample_size:,} keys from {table_name} & materializing to Pandas...")

# Chain ops: Sample random keys -> Join full table -> Count rows -> Pull to Pandas
dups = (
    t.inner_join(t.select(keys).distinct().order_by(ibis.random()).limit(sample_size), keys)
    .group_by(keys).aggregate(row_count=ibis._.count())
    .to_pandas()
)

print(f"✅ Sampled {len(dups):,} rows from {table_name}.")

if not dups.empty:
    print("📈 Rows per key distribution:\n", dups['row_count'].value_counts().sort_index().to_string())
    print("\n🔎 Top 5 Duplicate Keys:\n", dups.head(5).to_string(index=False))
    
    # Raw dump of the first duplicate found
    eg = dups.iloc[0]
    print(f"\n🔬 Raw Dump for {eg.registered_number} ({eg.year}):")
    dump = t.filter((t.registered_number == eg.registered_number) & (t.year == eg.year)).to_pandas()
    print(dump.dropna(axis=1, how='all').to_string(index=False))
else:
    print("✅ Zero duplicates found in the sample.")

🔍 Sampling 2,000 keys from fame_yearly & materializing to Pandas...


NameError: name 'df' is not defined

In [ ]:
import ibis

table_name = "fame_yearly"
dirs = get_data_dirs(segment="build")
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))

t = con.table(table_name)
keys = ["registered_number", "year"]
metric_cols = [c for c in t.columns if c not in keys]

print(f"🚀 Phase 1: Scanning {table_name} for clashing data across duplicates...")

# 1. Build a dictionary to count unique, non-null values for every single metric
conflict_aggs = {f"{c}_uniques": t[c].nunique() for c in metric_cols}

# Execute the grouping in DuckDB (Takes almost 0 RAM)
grouped = t.group_by(keys).aggregate(**conflict_aggs)

# 2. Build a dynamic Ibis filter: A row is a conflict if ANY metric has >1 unique value
conflict_condition = None
for c in metric_cols:
    col_cond = grouped[f"{c}_uniques"] > 1
    conflict_condition = col_cond if conflict_condition is None else (conflict_condition | col_cond)

conflicts_only = grouped.filter(conflict_condition)
conflict_count = conflicts_only.count().execute()

# 3. Hybrid Branching: Inspect OR Merge
if conflict_count > 0:
    print(f"⚠️ Found {conflict_count:,} firm-years where duplicate rows have DIFFERENT actual numbers!")
    print("📥 Pulling a 5-row sample into Pandas for inspection...\n")
    
    # Grab just 5 conflicting keys and join back to the main table to get the raw data
    sample_keys = conflicts_only.limit(5).select(keys)
    raw_conflicts = t.inner_join(sample_keys, keys).to_pandas()
    
    # Clean up the display so it fits on your screen
    raw_conflicts = raw_conflicts.dropna(axis=1, how='all').sort_values(keys)
    print("🔎 CONFLICT SAMPLE:")
    print(raw_conflicts.to_string(index=False))
    
    print("\n🛑 Stopping execution. Please review the clashing rows above.")
    print("If you want to forcefully merge them anyway (which will arbitrarily pick the MAX value),")
    print("you can run the merge logic in the block below.")

else:
    print("✅ No data conflicts found! All duplicates are perfectly sparse (max 1 true value per column).")
    print("🚀 Phase 2: Executing safe lossless merge...")
    
    # Safe to merge: max() will perfectly squash the sparse NULLs together
    merge_aggs = {col: t[col].max() for col in metric_cols}
    merged_table = t.group_by(keys).aggregate(**merge_aggs)
    
    # Write to a temporary table and swap (Atomic and safe)
    con.create_table(f"{table_name}_clean", merged_table, overwrite=True)
    con.drop_table(table_name)
    con.rename_table(f"{table_name}_clean", table_name)
    
    new_count = con.table(table_name).count().execute()
    print(f"✅ Successfully consolidated! New clean row count: {new_count:,}")

🚀 Phase 1: Scanning fame_yearly for clashing data across duplicates...


OutOfMemoryException: Out of Memory Error: could not allocate block of size 256.0 KiB (5.5 GiB/5.5 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

### [read] Check for duplicates in the resulting tables
Check for any duplicate registered_number values in fame_fixed and fame_derived  
Print a json in dirs.root_dir / "build" / "tmp" which is an array of objects, which contains properties:
- registered_number
- company_name
- industry_codes
- file_codes
- rows: the number of instances of this registered_number in this table
- all_other_properties_identical: a boolean indicating whether all other properties are identical across instances
- differing_properties: if and only iff all_other_properties_identical is False,
this will be a dict of the differing properties and their values across instances
In general within the for loop, use if not and then continue statements, rather than increasingly nested indentation

In [10]:
import ibis
import json
import pandas as pd
from prompt_toolkit import keys

from utils.f_0_dirs import get_data_dirs
from typing import Any

tables = ["fame_fixed", "fame_derived", "fame_yearly"]

dirs = get_data_dirs(segment="build")
db_path = dirs.output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(str(db_path))

time_start = pd.Timestamp.now()
def time_elapsed() -> str:
    elapsed = (pd.Timestamp.now() - time_start).total_seconds()
    return f"[{elapsed:5.1f}s] "

print(time_elapsed() + "⏱️ Starting duplicate detection and processing")

for table_name in tables:
    
    output_duplicate_paths = dirs.output_dir / f"duplicates_2_{table_name}.json"
    table = con.table(table_name)
    
    # 1. Identify columns to check
    ignore_cols = ["registered_number", "year"] if table_name == "fame_yearly" else ["registered_number"]
    check_cols = [c for c in table.columns if c not in ignore_cols]

    # 2. Build the Native Ibis Aggregation Dictionary
    aggs = {"rows": table.count()}
    for c in check_cols:
        # Cast to string, safely handle NULLs, and bundle into a DuckDB List
        aggs[f"{c}_vals"] = ibis.coalesce(table[c].cast("string"), "<NULL>").collect()

    print(f"{time_elapsed()}🚀 Executing native Ibis aggregation for {table_name}")

    # 3. Construct the Ibis Query Tree and Execute
    dup_counts = (
        table.group_by(ignore_cols)
        .aggregate(**aggs)
        .filter(ibis._.rows > 1)
    )
    count_dup_counts = dup_counts.count().execute()
    if count_dup_counts == 0:
        print(f"{time_elapsed()}✅ No duplicate ({', '.join(ignore_cols)}) values found in {table_name}.")
        continue
    print(f"{time_elapsed()}⚠️ Found {count_dup_counts:,} duplicate entries. Loading and processing")

    # Go back to the original table, throw away the 31.9 million healthy rows,
    # leaving ONLY the raw rows that belong to duplicates.
    filtered_dups = table.inner_join(dup_counts, ignore_cols)
    heavy_aggs = {}
    for c in check_cols:
        heavy_aggs[f"{c}_vals"] = ibis.coalesce(filtered_dups[c].cast("string"), "<NULL>").collect()
    heavy_agg = filtered_dups.group_by(ignore_cols + ["rows"]).aggregate(**heavy_aggs)
    grouped_df = heavy_agg.to_pyarrow().to_pandas()

    # 4. Fast Extraction: Iterate over the condensed array rows
    identical_list: list[dict[str, Any]] = []
    non_identical_list: list[dict[str, Any]] = []

    for _, row in grouped_df.iterrows():
        differing_properties = {}
        
        for col in check_cols:
            # We collected ALL values. We deduplicate them here in Python using set()
            # Because the arrays are tiny (usually 2-3 items), set() is functionally instant.
            unique_vals = list(set(row[f"{col}_vals"]))
            
            if len(unique_vals) > 1:
                # Convert back to standard None/str for clean JSON output
                clean_vals = [None if x == '<NULL>' else x for x in unique_vals]
                differing_properties[col] = clean_vals
                
        all_other_identical = len(differing_properties) == 0
        
        def get_summary_val(col_name) -> str | list | None:
            vals = row.get(col_name)
            if vals is None or len(vals) == 0:
                return None
            if len(vals) == 1:
                # If identical, just return the single string (or None)
                return None if pd.isna(vals[0]) else str(vals[0])
            else:
                # If they differ, return the array of strings so you can see both in the JSON
                return [None if pd.isna(x) else str(x) for x in vals]
        dup_record: dict[str, Any] = {
            "table": table_name,
            "registered_number": str(row["registered_number"])
        }
        if table_name == "fame_fixed":
            dup_record.update({
                "company_name": get_summary_val("company_name")
            })
        elif table_name == "fame_derived":
            dup_record.update({
                "industry_codes": get_summary_val("industry_codes"),
                "file_codes": get_summary_val("file_codes")
            })
        elif table_name == "fame_yearly":
            dup_record.update({
                "year": int(row["year"])
            })

        dup_record.update({
            "rows": int(row["rows"]),
            "all_other_properties_identical": all_other_identical,
            "differing_properties": differing_properties
        })
        
        if all_other_identical:
            identical_list.append(dup_record)
        else:
            non_identical_list.append(dup_record)

    # 5. Summary and Output
    print(f"{time_elapsed()}❌ Found {len(non_identical_list):,} duplicate entries in {table_name} with differing properties.")

    output_duplicates = non_identical_list + identical_list

    with open(output_duplicate_paths, "w") as f:
        json.dump(output_duplicates, f, indent=4, default=str)
        print(f"{time_elapsed()}✅ Listed {len(output_duplicates):,} duplicates, saved to: {output_duplicate_paths}")

[  0.0s] ⏱️ Starting duplicate detection and processing
[  0.0s] 🚀 Executing native Ibis aggregation for fame_fixed
[  1.6s] ⚠️ Found 587,295 duplicate entries. Loading and processing


RuntimeError: Query interrupted

# Understanding duplicates
- Listing categories of duplicates: let's start to understand how these might differ

In [4]:
import json
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="build")

output_dict = { "fame_fixed": {}, "fame_derived": {} }

for table_name in ["fame_fixed", "fame_derived"]:

    print(f"Analyzing duplicates for table: {table_name}")

    output_duplicate_paths = dirs.output_dir / f"duplicates_{table_name}.json"
    duplicates_data = []
    with open(output_duplicate_paths, "r") as f:
        duplicates_data = json.load(f)

    duplicates_ids = [entry["registered_number"] for entry in duplicates_data]
    duplicates_ids_set = set(duplicates_ids)
    if len(duplicates_ids) != len(duplicates_ids_set):
        raise ValueError(f"Duplicate registered_number values found in {output_duplicate_paths}. Please check the data.")
    else:
        print(f"{len(duplicates_ids_set):,} unique registered_number values found in {output_duplicate_paths}.")

    properties_set = set()
    keys_dict = {}
    for entry in duplicates_data:
        differing_props = entry.get("differing_properties", {})
        dp_frozen = frozenset(differing_props.keys())
        properties_set.add(dp_frozen)
        dp_key = ",".join(dp_frozen)
        if dp_key not in keys_dict:
            keys_dict[dp_key] = []
        keys_dict[dp_key].append(entry["registered_number"])

    properties_list = sorted(list(properties_set), key=lambda x: (len(x), x))  # Sort by length first, then alphabetically
    # True for i, x in enumerate(properties_list) if i == 0 or len(x) > len(properties_list[i - 1]) else False
    start_of_increment = set({ x for i, x in enumerate(properties_list) if i == 0 or len(x) > len(properties_list[i - 1]) })
    print(f"--- Unique properties ({len(properties_list)}): {properties_list}")
    for frozen_p in properties_list:
        if frozen_p in start_of_increment:
            print(f"--- Dupe categories: {len(frozen_p)}")
        reg_nums = keys_dict.get(",".join(frozen_p), [])
        print(f"--- --- ({len(reg_nums):,}) {','.join(frozen_p)}: {', '.join(reg_nums)}")
        output_dict[table_name][",".join(frozen_p)] = reg_nums


Analyzing duplicates for table: fame_fixed
6,742 unique registered_number values found in C:\Users\lazyst\Files\ucl\Dissertation\build\output\duplicates_fame_fixed.json.
--- Unique properties (450): [frozenset({'guo'}), frozenset({'primary_trading_address'}), frozenset({'branch_name'}), frozenset({'no_of_available_years'}), frozenset({'entity_type'}), frozenset({'company_name'}), frozenset({'guo_nb'}), frozenset({'latest_accounts_date'}), frozenset({'guo', 'guo_nb'}), frozenset({'primary_trading_address', 'latest_accounts_date'}), frozenset({'primary_trading_address', 'guo_nb'}), frozenset({'entity_type', 'latest_accounts_date'}), frozenset({'primary_trading_address', 'branch_name'}), frozenset({'primary_trading_address', 'guo'}), frozenset({'guo', 'latest_accounts_date'}), frozenset({'no_of_available_years', 'latest_accounts_date'}), frozenset({'ro_address', 'ro_address_line_2'}), frozenset({'branch_name', 'latest_accounts_date'}), frozenset({'primary_uk_sic_2007_description', 'primar

### [write] Start clearing files that are errored from main_dedupe

In [7]:
import ibis

for table_name in ["fame_fixed", "fame_derived"]:
    reg_nums_to_clear: list[str] = output_dict[table_name].get("", [])
    print(f"Fetched {len(reg_nums_to_clear):,} unique duplicate IDs to resolve")

    if not reg_nums_to_clear:
        print(f"No rows to clear from {table_name} with no differing properties.")
        continue

    con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
    table = con.table(table_name)
    rows_count_start = table.count().execute()
    print(f"Starting with {rows_count_start:,} rows in {table_name}") 

    # 1. Convert the Python list to an Ibis MemTable
    target_ids = ibis.memtable({"registered_number": reg_nums_to_clear})

    # 2. STREAM A: All rows NOT in our target list (Keep exactly as they are)
    unaffected_rows = table.anti_join(target_ids, "registered_number")

    # 3. STREAM B: All rows IN our target list, collapsed to exactly one instance.
    # Because you confirmed these have "no differing properties", .distinct() is perfectly safe to use here.
    resolved_duplicates = table.inner_join(target_ids, "registered_number").distinct()

    # Safety Check: Did distinct() collapse them to exactly one per ID?
    resolved_count = resolved_duplicates.count().execute()
    if resolved_count != len(reg_nums_to_clear):
        raise ValueError(f"Wait! Expected exactly {len(reg_nums_to_clear):,} distinct rows, but got {resolved_count:,}. Some duplicates might have differing properties!")

    # 4. Combine the streams
    clean_table = unaffected_rows.union(resolved_duplicates)

    rows_count_clean = clean_table.count().execute()
    rows_removed = rows_count_start - rows_count_clean

    print(f"Net duplicate rows removed: {rows_removed:,}")
    if rows_removed < 0:
        continue  # When we run this code multiple times, we skip. No need to risk overwriting the table

    print(f"Preparing to overwrite {table_name} with {rows_count_clean:,} rows...")

    # 5. Execute the overwrite
    con.create_table(f"{table_name}_cleaned", clean_table, overwrite=True)
    con.create_table(table_name, con.table(f"{table_name}_cleaned"), overwrite=True)
    con.drop_table(f"{table_name}_cleaned")

    rows_count_end = con.table(table_name).count().execute()

    print(f"✅ {table_name} had {rows_count_start:,} rows before clearing, now has {rows_count_end:,} rows.")

Fetched 0 unique duplicate IDs to resolve
No rows to clear from fame_fixed with no differing properties.
Fetched 47,809 unique duplicate IDs to resolve
Starting with 7,948,736 rows in fame_derived


ValueError: Wait! Expected exactly 47,809 distinct rows, but got 56,763. Some duplicates might have differing properties!

### In fame_yearly

In [8]:
# --- Configuration ---
table_name = "fame_yearly"
t = con.table(table_name)
keys = ["registered_number", "year"]
metric_cols = [c for c in t.columns if c not in keys]

# --- 1. Pre-Check ---
old_count = t.count().execute()
print(f"🚀 Starting consolidation for '{table_name}'")
print(f"📊 Starting row count: {old_count:,}")
print(f"⚙️  Merging split rows using MAX() across {len(metric_cols)} columns...")

# --- 2. Build Aggregation ---
merge_aggs = {col: t[col].max() for col in metric_cols}
merged_table = t.group_by(keys).aggregate(**merge_aggs)

# --- 3. Execute & Write ---
print("💾 Writing to temporary table (this relies entirely on DuckDB's C++ engine)...")
con.create_table(f"{table_name}_clean", merged_table, overwrite=True)

# --- 4. Atomic Swap ---
print("🔄 Swapping clean table into place...")
con.drop_table(table_name)
con.create_table(table_name, con.table(f"{table_name}_clean"), overwrite=True)

# --- 5. Post-Check & Summary ---
new_count = con.table(table_name).count().execute()
rows_removed = old_count - new_count

print(f"✅ Consolidation complete!")
print(f"📉 New row count: {new_count:,}")
print(f"🗑️  Duplicate rows squashed: {rows_removed:,}")

🚀 Starting consolidation for 'fame_yearly'
📊 Starting row count: 45,103,733
⚙️  Merging split rows using MAX() across 35 columns...
💾 Writing to temporary table (this relies entirely on DuckDB's C++ engine)...
🔄 Swapping clean table into place...
✅ Consolidation complete!
📉 New row count: 40,846,539
🗑️  Duplicate rows squashed: 4,257,194


In [ ]:
import ibis

t = con.table("fame_derived")
t_count = t.count().execute()

# 1. Inspect the table dynamically (No hardcoding schema names!)
concat_cols = ["industry_codes", "file_codes"]
standard_cols = [c for c in t.columns if c not in concat_cols and c != "registered_number"]

# 2. Tell DuckDB how to merge the standard identical columns.
# Taking .max() safely grabs the identical value across the duplicate rows.
aggs = {c: t[c].max() for c in standard_cols}
clean_table = t.group_by("registered_number").aggregate(**aggs)

# 3. Safely concatenate the array columns without duplicating tags (e.g., avoiding "01,01")
for col in concat_cols:
    unique_str_agg = (
        t.select("registered_number", col)
        .filter(t[col] != "")
        .distinct()  # Drops duplicates before concatenating
        .group_by("registered_number")
        .aggregate(**{col: ibis._[col].group_concat(",")})
    )
    # Join it back onto our clean base table
    clean_table = clean_table.left_join(unique_str_agg, "registered_number").drop("registered_number_right")

# 4. Execute the replacement safely
con.create_table("fame_derived_clean", clean_table, overwrite=True)
con.create_table("fame_derived", con.table("fame_derived_clean"), overwrite=True)
con.drop_table("fame_derived_clean")

print(f"✅ Deduplicated fame_derived dynamically removing {t_count - clean_table.count().execute()} rows")

**Companies outside the UK.** We focus on companies that are incorporated in the UK.
We exclude accounts of non-UK companies. The FAME data includes some consolidated
accounts for non-UK firms that can be identified through additional letters in the registered
number (i.e. firm ID), which starts with either ‘#’ (e.g. Spain, Germany), ‘IE’ (i.e.
Ireland), or ‘GI’ (i.e. Gibraltar). We therefore keep those in England, Northern Ireland,
Scotland, and Wales.

In [ ]:
# Fetch all companies that have letters as their first two characters of their registered number
# From the fame_
